# Spreadsheets end to end

Loading an `.xlsx` with `read_excel`, working on it as a DataFrame, then
handing the numeric columns to the natural-language layer.

## Before running

From the repo root:

```sh
pip install -e ".[all,dev]"
pip install python-dotenv
```

`[all]` brings the provider SDKs. Installing only `[excel,dev]` leaves the
model cells failing with *"Please install google-genai"*, because the
provider package ships separately from `pydantic-ai` itself.

Then put a key in `examples/.env`, matching whichever model you use:

```
GEMINI_API_KEY=...
```

Cells up to *Asking questions* need no key and no provider SDK.

In [1]:
from dotenv import load_dotenv

load_dotenv(".env")

import numpy as np
import numpyai_dashboard as npi

## Loading

`read_excel` returns a `pandas.DataFrame`. Every column is kept, with its type
inferred by the Rust reader.

In [2]:
df = npi.read_excel("sample_sales.xlsx")
df.head()

,region,rep,product,order_date,units,unit_price,discount,closed,notes
0,EMEA,R. Ahmed,Basic,2023-01-01,21.0,19.99,NaN,False,renewal
1,APAC,R. Brown,Pro,2023-01-03,26.0,49.50,0.20,True,NaN
2,AMER,R. Chen,Enterprise,2023-01-05,5.0,199.00,0.25,True,NaN
3,LATAM,R. Duarte,Basic,2023-01-07,7.0,19.99,0.11,False,NaN
4,EMEA,R. Eriksen,Pro,2023-01-09,4.0,49.50,0.27,True,renewal


In [3]:
df.dtypes

region                   str
rep                      str
product                  str
order_date    datetime64[ms]
units                float64
unit_price           float64
discount             float64
closed                  bool
notes                    str
dtype: object

Text stays text, dates become `datetime64`, `TRUE`/`FALSE` become `bool`, and
blank cells become the null of whichever type the column is. Nothing is dropped.

In [4]:
df[["discount", "notes"]].isna().sum()

discount     14
notes       112
dtype: int64

### Options

Pick a sheet by name or index, skip the header row, or read only the first
`n_rows` when you just want to look at the shape of a large file.

In [5]:
npi.read_excel("sample_sales.xlsx", sheet="Sales", n_rows=5)

,region,rep,product,order_date,units,unit_price,discount,closed,notes
0,EMEA,R. Ahmed,Basic,2023-01-01,21.0,19.99,NaN,False,renewal
1,APAC,R. Brown,Pro,2023-01-03,26.0,49.50,0.20,True,NaN
2,AMER,R. Chen,Enterprise,2023-01-05,5.0,199.00,0.25,True,NaN
3,LATAM,R. Duarte,Basic,2023-01-07,7.0,19.99,0.11,False,NaN
4,EMEA,R. Eriksen,Pro,2023-01-09,4.0,49.50,0.27,True,renewal


With `header=False` the first row is data and columns are positional.

In [6]:
npi.read_excel("sample_sales.xlsx", header=False).head(3)

,col0,col1,col2,col3,col4,col5,col6,col7,col8
0,region,rep,product,order_date,units,unit_price,discount,closed,notes
1,EMEA,R. Ahmed,Basic,2023-01-01 00:00:00,21,19.99,NaN,false,renewal
2,APAC,R. Brown,Pro,2023-01-03 00:00:00,26,49.5,0.2,true,NaN


Sheets can be chosen by index too, and unsupported formats fail loudly
rather than half-working.

In [7]:
from numpyai_dashboard._exceptions import NumpyAIError

print(npi.read_excel("sample_sales.xlsx", sheet=0).shape)

try:
    npi.read_excel("notes.csv")
except NumpyAIError as exc:
    print("error:", exc)

(150, 9)
error: read_excel cannot read .csv files. CSV is not supported yet.


## Working in pandas

Ordinary DataFrame work. Note `discount` has blanks, so fill them before
arithmetic.

In [8]:
df["revenue"] = df["units"] * df["unit_price"] * (1 - df["discount"].fillna(0))

df.groupby("region")["revenue"].sum().sort_values(ascending=False).round(2)

region
APAC     92209.21
EMEA     84916.77
LATAM    81800.69
AMER     75051.67
Name: revenue, dtype: float64

`order_date` is a real `datetime64`, so date filtering works without parsing.

In [9]:
q1 = df[df["order_date"] < "2023-04-01"]
print(f"{len(q1)} orders in Q1")
q1.groupby("product")["revenue"].sum().round(2)

45 orders in Q1


product
Basic          6284.66
Enterprise    77178.17
Pro           18788.72
Name: revenue, dtype: float64

## Handing off to NumPy

`to_numpy()` on numeric columns is effectively free. Pass `columns=` so the
model knows what each column means rather than seeing bare indices.

In [10]:
cols = ["units", "unit_price", "discount", "revenue"]

arr = npi.array(df[cols].to_numpy(), columns=cols)
arr

numpyai_dashboard.array(shape=(150, 4), dtype=float64)

The wrapper forwards to the underlying array, so ordinary NumPy still works.

In [11]:
print(arr.shape, arr.dtype)
print("column means:", arr.mean(axis=0))
print("first 3 rows:\n", arr.data[:3])

(150, 4) float64
column means: numpyai_dashboard.array(shape=(4,), dtype=float64)
first 3 rows:
 [[2.1000e+01 1.9990e+01        nan 4.1979e+02]
 [2.6000e+01 4.9500e+01 2.0000e-01 1.0296e+03]
 [5.0000e+00 1.9900e+02 2.5000e-01 7.4625e+02]]


Arithmetic returns a new wrapped array.

In [12]:
doubled = arr * 2
doubled

numpyai_dashboard.array(shape=(150, 4), dtype=float64)

## Asking questions

Everything below needs a provider key. Generated code is syntax-checked and
independently judged before a result comes back.

In [13]:
arr.chat("What is the mean revenue?")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: What is the mean revenue?                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(2226.522308666667)

In [14]:
arr.chat("Correlation between units and revenue.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Correlation between units and revenue.                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(0.43744500530767105)

Missing values are visible to the model, so it can be asked about them directly.

In [15]:
arr.chat("How many rows have a missing discount?")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: How many rows have a missing discount?                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.int64(14)

Because `columns=` was passed, questions can name columns directly.

In [16]:
arr.chat("Mean units for rows where the discount is above 0.2.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Mean units for rows where the discount is above 0.2.                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(29.91176470588235)

A query can return an array rather than a scalar.

In [17]:
imputed = arr.chat("Replace missing discounts with the column mean.")
imputed

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Replace missing discounts with the column mean.                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

array([[2.10000000e+01, 1.99900000e+01, 1.40294118e-01, 4.19790000e+02],
       [2.60000000e+01, 4.95000000e+01, 2.00000000e-01, 1.02960000e+03],
       [5.00000000e+00, 1.99000000e+02, 2.50000000e-01, 7.46250000e+02],
       [7.00000000e+00, 1.99900000e+01, 1.10000000e-01, 1.24537700e+02],
       [4.00000000e+00, 4.95000000e+01, 2.70000000e-01, 1.44540000e+02],
       [3.00000000e+00, 1.99000000e+02, 3.00000000e-02, 5.79090000e+02],
       [2.70000000e+01, 1.99900000e+01, 2.00000000e-02, 5.28935400e+02],
       [6.00000000e+00, 4.95000000e+01, 1.70000000e-01, 2.46510000e+02],
       [4.00000000e+00, 1.99000000e+02, 2.50000000e-01, 5.97000000e+02],
       [1.50000000e+01, 1.99900000e+01, 1.90000000e-01, 2.42878500e+02],
       [3.80000000e+01, 4.95000000e+01, 2.80000000e-01, 1.35432000e+03],
       [3.70000000e+01, 1.99000000e+02, 1.40294118e-01, 7.36300000e+03],
       [3.80000000e+01, 1.99900000e+01, 1.20000000e-01, 6.68465600e+02],
       [3.00000000e+00, 4.95000000e+01, 1.70000000e

## Several arrays at once

`NumpyAISession` exposes each array to the model as `arr1`, `arr2`, ...

In [18]:
emea = df.loc[df["region"] == "EMEA", cols].to_numpy()
apac = df.loc[df["region"] == "APAC", cols].to_numpy()

sess = npi.NumpyAISession([emea, apac])
sess.chat("Compare the mean revenue of the two arrays.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Compare the mean revenue of the two arrays.                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

np.float64(-45.75129045438052)

## Diagnosis

Suggests analysis steps for the data rather than computing an answer.

In [19]:
diag = npi.Diagnosis(sess)
diag.steps(task="Give me 5 steps to analyse this sales data.")

────────────────────────────────────────────────── LLM Response ───────────────────────────────────────────────────

╭────────────────────────────────────────────── Data Analysis Steps ──────────────────────────────────────────────╮
│ [ "First, confirm the dimensions of arr1 and arr2 using shape and ndim. Then, for each array, use isnan to      │
│ identify all missing values. Sum these boolean masks per column to quantify the number of NaNs. For the column  │
│ representing discount rates (column index 2), impute missing values with the nanmedian of that column to        │
│ minimize the effect of potential outliers, or replace with 0 if a missing discount implies no discount. For the │
│ column representing total revenue (column index 3 for arr1), if there are only a few NaNs, consider removing    │
│ those rows; otherwise, also impute with nanmedian. This step establishes a clean and consistent dataset,        │
│ diagnosing data integrity and preparing it for quantitative analysis.", "Next, calculate comprehensive          │
│ descriptive statistics for each numerical column in both arr1 and arr2 (after missing value treatment). This    │
│ involves computing the nanmean, nanmedian, nanstd, nanmin, nanmax, and the 25th and 75th nanpercentile (Q1 and  │
│ Q3) for each column. These statistics will summarize the central tendency, dispersion, and range of values.     │
│ This step diagnoses the overall characteristics of each variable, identifying average values, spread, and       │
│ potential skewness by comparing the mean and median, providing a foundational understanding of the data's       │
│ distribution.", "Identify potential outliers in the 'total_revenue' column (index 3) and possibly 'quantity'    │
│ (index 0) for both arrays. Use the Interquartile Range (IQR) method: any value below Q1 - 1.5 * IQR or above Q3 │
│ + 1.5 * IQR is considered an outlier. Based on domain knowledge, decide on an appropriate handling strategy.    │
│ Options include capping these extreme values using clip to the calculated upper or lower bounds if they are     │
│ deemed erroneous or unduly influential, or applying a mathematical log transformation to the entire column to   │
│ reduce their impact if they represent legitimate but skewed data points. This step diagnoses data points that   │
│ might disproportionately influence analysis and ensures a more robust dataset for further calculations.",       │
│ "Perform a correlation analysis by computing the pairwise corrcoef matrix for all numerical columns within each │
│ processed array. Analyze the strength and direction of linear relationships, especially focusing on how         │
│ 'quantity' (column 0), 'price_per_unit' (column 1), and 'discount_rate' (column 2) correlate with               │
│ 'total_revenue' (column 3). This step diagnoses the interdependencies between different aspects of the sales    │
│ data, helping to understand which factors move together or in opposition, and providing insights into potential │
│ drivers of sales figures.", "Finally, conduct a comparative analysis by contrasting the descriptive statistics, │
│ outlier patterns, and correlation matrices between arr1 and arr2 to identify significant differences or         │
│ similarities. Additionally, create a new feature for both arrays: 'net_price_per_unit', calculated as           │
│ price_per_unit * (1 - discount_rate), and append this new column. This step directly diagnoses the distinctions │
│ between the two datasets and provides enhanced features for more nuanced interpretation. For example, comparing │
│ 'net_price_per_unit' between the arrays can reveal differences in pricing strategies or customer segments." ]   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

['First, confirm the dimensions of `arr1` and `arr2` using `shape` and `ndim`. Then, for each array, use `isnan` to identify all missing values. Sum these boolean masks per column to quantify the number of NaNs. For the column representing discount rates (column index 2), impute missing values with the `nanmedian` of that column to minimize the effect of potential outliers, or replace with 0 if a missing discount implies no discount. For the column representing total revenue (column index 3 for `arr1`), if there are only a few NaNs, consider removing those rows; otherwise, also impute with `nanmedian`. This step establishes a clean and consistent dataset, diagnosing data integrity and preparing it for quantitative analysis.',
 "Next, calculate comprehensive descriptive statistics for each numerical column in both `arr1` and `arr2` (after missing value treatment). This involves computing the `nanmean`, `nanmedian`, `nanstd`, `nanmin`, `nanmax`, and the 25th and 75th `nanpercentile` (Q1 

## Verbose mode

`verbose=True` prints every intermediate step: the generated code, the
judgement, and any retries.

In [20]:
loud = npi.array(df[cols].to_numpy(), columns=cols, verbose=True)
loud.chat("Total revenue where units exceed 40.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Total revenue where units exceed 40.                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Attempt 1/3...

╭──────────────────────────────────────────────── Generated Code ─────────────────────────────────────────────────╮
│   1 cleaned_revenue = np.nan_to_num(arr[:, 3])                                                                  │
│   2 cleaned_units = np.nan_to_num(arr[:, 0])                                                                    │
│   3                                                                                                             │
│   4 condition = cleaned_units > 40                                                                              │
│   5 output = np.sum(cleaned_revenue[condition])                                                                 │
│   6 metadata = "total revenue where units exceed 40"                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Executing generated code...

159278.66410000002

total revenue where units exceed 40

╭─────────────────────────────────────────────────── Judgment ────────────────────────────────────────────────────╮
│ ✓ correctly interprets the query                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Judgment passed!

np.float64(159278.66410000002)

## Choosing a model

Any Pydantic AI model spec works. The default is `google:gemini-2.5-flash`.

In [21]:
# needs the matching extra installed, e.g. numpyai-dashboard[anthropic]
# alt = npi.array(df[cols].to_numpy(), columns=cols, model="anthropic:claude-sonnet-4-5")
# alt.chat("Median units.")

## The judge rejects non-answers

Ask for something the data cannot support and the model will often hand back a
polite explanation rather than a computation. The judge is what stops that from
counting as an answer: it compares the generated code against the query and
rejects prose standing in for a result.

With `max_tries=1` there is no retry, so `chat` prints the failures, warns, and
returns `None`. It does not raise, so one bad question will not end a session.
Raise `max_tries` to watch it retry with the rejection fed back in.

In [22]:
stubborn = npi.array(df[cols].to_numpy(), columns=cols, max_tries=1)
stubborn.chat("Plot a pie chart of customer sentiment.")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Query: Plot a pie chart of customer sentiment.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Attempt 1/1...

╭──────────────────────────────────────────────── Generated Code ─────────────────────────────────────────────────╮
│   1 # The array 'arr' does not contain customer sentiment data,                                                 │
│   2 # which is required to plot a pie chart of customer sentiment.                                              │
│   3 # Therefore, the requested operation cannot be performed.                                                   │
│   4 output = None                                                                                               │
│   5 metadata = "Operation failed: Customer sentiment data is missing in the array 'arr'."                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Executing generated code...

✗ Attempt 1 failed: execution returned None

                  Error Details                  
╔═════════╤═════════════════════════════════════╗
║ Attempt │ Error                               ║
╟─────────┼─────────────────────────────────────╢
║ 1       │ Try 1: Code execution returned None ║
╚═════════╧═════════════════════════════════════╝

/tmp/ipykernel_161184/140897066.py:2: UserWarning: Validation failed after 1 attempts. Please check the validity of the code.
  stubborn.chat("Plot a pie chart of customer sentiment.")
